# Project 2B: Foothold EKF on RealReplay

This notebook runs the foothold-augmented EKF on a selected bag in `data/` (start with `building.bag`).


In [ ]:
import numpy as np
import gtsam
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from measurement import Measurement
from real_replay import RealReplay
from foothold_filter import FootholdNavStateImuEKF


In [ ]:
cm = 0.01
APPLY_HEIGHT_PRIOR = True  # Keep False for building.bag (Tasks 3-4). Task 6 (stair.bag): run once with True, once with False.
bag_path = "data/stair.bag"
# bag_path = "data/building.bag"
r = RealReplay(bag_path)

foot_names = r.foot_names
x0 = gtsam.NavState(gtsam.Rot3(), np.zeros(3), np.zeros(3))
rot_var_init = np.array([1e-2, 1e-2, 1e-6], dtype=float)
P0 = np.diag(
    np.concatenate([rot_var_init, [0.05] * 3, [0.05] * 3, [0.08] * (3 * len(foot_names))])
)

ekf = FootholdNavStateImuEKF(
    x0=x0,
    P0=P0,
    num_feet=len(foot_names),
    foot_names=foot_names,
    sigma_gyro=8e-4,
    sigma_acc=1e-2,
    foothold_process_sigma=1e-4,
    sigma_meas=1 * cm,
    sigma_height_prior=15 * cm,
)

landmark_est = []
p_hist = []
ypr_hist = []
R_hist = []
dt_hist = []
timestamp_hist = []
forward_vel_hist = []


def ekf_callback(m: Measurement) -> None:
    if m.dt > 0.0:
        ekf.predict(m.omega_meas, m.f_meas, m.dt)
    ekf.process_contact_measurements(
        m.old_contacts,
        m.new_contacts,
        z_world=0.0,
        apply_height_prior=APPLY_HEIGHT_PRIOR,
    )

    for foot_id in sorted(m.new_contacts):
        event_label = f"{foot_id}_{m.k}"
        landmark_est.append((event_label, ekf.footholds[ekf.resolve_index(foot_id)].copy()))

    p_hist.append(np.asarray(ekf.x.position(), dtype=float))
    ypr_hist.append(np.asarray(ekf.x.attitude().ypr(), dtype=float))
    R_hist.append(np.asarray(ekf.x.attitude().matrix(), dtype=float))
    dt_hist.append(float(m.dt))

    if m.timestamp_s is not None:
        timestamp_hist.append(float(m.timestamp_s))
    elif timestamp_hist:
        timestamp_hist.append(timestamp_hist[-1] + float(m.dt))
    else:
        timestamp_hist.append(0.0)

    forward_vel_hist.append(
        float((ekf.x.attitude().matrix().T @ np.asarray(ekf.x.velocity(), dtype=float))[0])
    )


r.replay(ekf_callback)
r.close()

p_hist = np.asarray(p_hist)
ypr_hist = np.asarray(ypr_hist)
R_hist = np.asarray(R_hist)
dt_hist = np.asarray(dt_hist)
timestamp_hist = np.asarray(timestamp_hist)
forward_vel_hist = np.asarray(forward_vel_hist)

print(f"Foothold-EKF samples: {len(p_hist)}")
print(f"Foothold initializations: {len(landmark_est)}")


In [ ]:
k = np.arange(len(p_hist))

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Position (x,y,z)", "Yaw-Pitch-Roll", "Forward velocity"))

fig.add_trace(go.Scatter(x=k, y=p_hist[:, 0], name="x"), row=1, col=1)
fig.add_trace(go.Scatter(x=k, y=p_hist[:, 1], name="y"), row=1, col=1)
fig.add_trace(go.Scatter(x=k, y=p_hist[:, 2], name="z"), row=1, col=1)

fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 0], name="yaw"), row=2, col=1)
fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 1], name="pitch"), row=2, col=1)
fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 2], name="roll"), row=2, col=1)

fig.add_trace(go.Scatter(x=k, y=forward_vel_hist, name="forward vel"), row=3, col=1)

fig.update_layout(height=800, width=1000, title=f"Foothold EKF on {bag_path}, height prior {'on' if APPLY_HEIGHT_PRIOR else 'off'}")
fig.update_xaxes(title_text="sample index", row=3, col=1)
fig.show()


In [ ]:
if landmark_est:
    # Color order requested: red, green, blue, then a fourth distinct color.
    palette = ["#d62728", "#2ca02c", "#1f77b4", "#ff7f0e"]
    ordered_feet = list(foot_names)
    color_map = {foot: palette[i % len(palette)] for i, foot in enumerate(ordered_feet)}

    fig = go.Figure()
    for foot in ordered_feet:
        pts = np.asarray(
            [p for label, p in landmark_est if label.startswith(f"{foot}_")],
            dtype=float,
        )
        if pts.size == 0:
            continue
        fig.add_trace(
            go.Scatter(
                x=pts[:, 0],
                y=pts[:, 1],
                mode="markers",
                marker=dict(size=7, color=color_map[foot]),
                name=foot,
            )
        )

    fig.update_layout(
        height=500,
        width=900,
        title="Estimated footholds at contact starts",
        xaxis_title="x [m]",
        yaxis_title="y [m]",
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()
else:
    print("No foothold start events found.")


## Velodyne 3D Map in World Frame

This section associates each `/velodyne_points` cloud with the nearest EKF pose in time and transforms the cloud into the world frame.

In [ ]:
from pathlib import Path

from rosbags.highlevel import AnyReader


def pointcloud2_xyz(msg) -> np.ndarray:
    """Decode XYZ points from a sensor_msgs/PointCloud2 message."""
    n = int(msg.width) * int(msg.height)
    if n == 0:
        return np.empty((0, 3), dtype=np.float32)

    offsets = {field.name: int(field.offset) for field in msg.fields}
    for key in ("x", "y", "z"):
        if key not in offsets:
            raise ValueError(f"PointCloud2 message is missing '{key}' field.")

    endian = ">" if bool(msg.is_bigendian) else "<"
    dtype = np.dtype(
        {
            "names": ["x", "y", "z"],
            "formats": [f"{endian}f4", f"{endian}f4", f"{endian}f4"],
            "offsets": [offsets["x"], offsets["y"], offsets["z"]],
            "itemsize": int(msg.point_step),
        }
    )
    arr = np.frombuffer(msg.data, dtype=dtype, count=n)
    xyz = np.column_stack((arr["x"], arr["y"], arr["z"])).astype(np.float32, copy=False)
    return xyz[np.all(np.isfinite(xyz), axis=1)]


def ros_time_to_s(time_msg) -> float | None:
    if time_msg is None:
        return None
    sec = getattr(time_msg, "sec", None)
    nanosec = getattr(time_msg, "nanosec", None)
    if sec is None or nanosec is None:
        return None
    sec_i = int(sec)
    nsec_i = int(nanosec)
    if sec_i == 0 and nsec_i == 0:
        return None
    return float(sec_i) + float(nsec_i) * 1e-9


def message_time_s(msg, record_time_s: float) -> tuple[float, bool]:
    for field in (
        "sensor_timestamp",
        "acquisition_timestamp",
        "measurement_timestamp",
        "source_timestamp",
    ):
        ts = ros_time_to_s(getattr(msg, field, None))
        if ts is not None:
            return ts, True

    ts = ros_time_to_s(getattr(getattr(msg, "header", None), "stamp", None))
    if ts is not None:
        return ts, True

    return float(record_time_s), False


pose_t_s = np.asarray(timestamp_hist, dtype=float)
if pose_t_s.size == 0:
    raise ValueError("No EKF poses available for map building.")

# Assumed lidar-to-body extrinsics (identity). Replace with calibration if available.
R_body_velodyne = np.eye(3, dtype=float)
t_body_velodyne = np.zeros(3, dtype=float)

MAP_POINTS_PER_FRAME = 120
VELODYNE_TOPIC = "/velodyne_points"

bag_file = Path(bag_path)
if not bag_file.exists():
    bag_file = Path("project_2b") / bag_path

if not bag_file.exists():
    raise FileNotFoundError(f"Bag file not found: {bag_path}")

world_chunks = []
frame_count = 0
used_frames = 0
sensor_time_frames = 0
record_time_fallback_frames = 0

with AnyReader([bag_file]) as reader:  # type: ignore[arg-type]
    velodyne_connections = [c for c in reader.connections if c.topic == VELODYNE_TOPIC]
    if not velodyne_connections:
        raise ValueError(f"Topic '{VELODYNE_TOPIC}' not found in {bag_file}.")

    for connection, record_time_ns, rawdata in reader.messages(connections=velodyne_connections):
        msg = reader.deserialize(rawdata, connection.msgtype)
        record_time_s = float(record_time_ns) * 1e-9
        t_cloud, used_sensor_time = message_time_s(msg, record_time_s)

        if used_sensor_time:
            sensor_time_frames += 1
        else:
            record_time_fallback_frames += 1

        idx = int(np.searchsorted(pose_t_s, t_cloud, side="left"))
        if idx >= len(pose_t_s):
            idx = len(pose_t_s) - 1
        if idx > 0 and abs(pose_t_s[idx - 1] - t_cloud) < abs(pose_t_s[idx] - t_cloud):
            idx -= 1

        xyz_velodyne = pointcloud2_xyz(msg)
        if xyz_velodyne.size == 0:
            frame_count += 1
            continue

        stride = max(1, xyz_velodyne.shape[0] // MAP_POINTS_PER_FRAME)
        xyz_velodyne = xyz_velodyne[::stride]

        xyz_body = (R_body_velodyne @ xyz_velodyne.T).T + t_body_velodyne
        xyz_world = (R_hist[idx] @ xyz_body.T).T + p_hist[idx]

        world_chunks.append(xyz_world.astype(np.float32, copy=False))
        frame_count += 1
        used_frames += 1

map_world = np.concatenate(world_chunks, axis=0) if world_chunks else np.empty((0, 3), dtype=np.float32)
traj_stride = max(1, len(p_hist) // 5000)

In [ ]:
print(f"Velodyne frames processed: {frame_count}")
print(f"Velodyne frames used: {used_frames}")
print(f"Map points rendered: {len(map_world):,}")
print(f"Cloud timestamps from sensor/header time: {sensor_time_frames}")
print(f"Cloud timestamps from bag-record fallback: {record_time_fallback_frames}")
print(
    "Assumption: velodyne frame is co-located with robot body frame (identity extrinsics)."
)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=map_world[:, 0],
        y=map_world[:, 1],
        z=map_world[:, 2],
        mode="markers",
        marker=dict(
            size=1.5, color=map_world[:, 2], colorscale="Viridis", opacity=0.45
        ),
        showlegend=False,
        name="Velodyne map",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=p_hist[::traj_stride, 0],
        y=p_hist[::traj_stride, 1],
        z=p_hist[::traj_stride, 2],
        mode="lines",
        line=dict(color="#d62728", width=5),
        showlegend=False,
        name="Foothold-EKF trajectory",
    )
)

fig.update_layout(
    height=900,
    width=1450,
    margin=dict(l=0, r=0, b=0, t=0),
    title="Global 3D map from Velodyne + Foothold-EKF trajectory",
    scene=dict(
        xaxis_title="x [m]",
        yaxis_title="y [m]",
        zaxis_title="z [m]",
        aspectmode="data",
    ),
)
fig.show()


In [ ]:
from pathlib import Path

try:
    import laspy
except ImportError as exc:
    raise ImportError(
        "Saving to .laz requires laspy with a LASzip backend. Install with: pip install laspy lazrs"
    ) from exc

if "map_world" not in globals() or map_world.size == 0:
    raise ValueError("map_world is empty. Run the Velodyne map-building cell first.")

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
output_laz = output_dir / "foothold_ekf_velodyne_map.laz"

xyz = np.asarray(map_world, dtype=np.float64)
header = laspy.LasHeader(point_format=3, version="1.4")
header.scales = np.array([0.001, 0.001, 0.001], dtype=np.float64)  # 1 mm
header.offsets = np.min(xyz, axis=0)

las = laspy.LasData(header)
las.x = xyz[:, 0]
las.y = xyz[:, 1]
las.z = xyz[:, 2]

# Optional intensity from normalized height (0..65535) for better viewer coloring defaults.
z_norm = xyz[:, 2] - np.min(xyz[:, 2])
if np.max(z_norm) > 0:
    z_norm = z_norm / np.max(z_norm)
las.intensity = np.round(z_norm * np.iinfo(np.uint16).max).astype(np.uint16)

las.write(output_laz)
print(f"Saved LAZ map: {output_laz.resolve()}")
print(f"Points saved: {xyz.shape[0]:,}")
